# 📚 Retrievers in LangChain: The Smart Librarian
- A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them.

- The document retriever is `responsible for finding relevant information from a large corpus of documents based on the input question using semantic search`. This information is then passed to the LLM, which generates a response. The unique aspect of RAG is the way it combines these two components.

In [ ]:
!pip install langchain chromadb faiss-cpu langchain-huggingface langchain-community wikipedia langchain_chroma

# Import Libraries

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEmbeddings

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [50]:
from langchain_community.retrievers import WikipediaRetriever

In [6]:
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

In [7]:
from langchain_core.documents import Document

# Model Load

In [8]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('unsloth/gemma-3-1b-it')
llm = AutoModelForCausalLM.from_pretrained('unsloth/gemma-3-1b-it')

# Build pipeline
pipe = pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer
)

# Wrap in LangChain pipeline
llm_pipeline = HuggingFacePipeline(pipeline=pipe)

# Build Chat model
llm_model = ChatHuggingFace(llm=llm_pipeline)

# Build embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},   # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Utils

In [26]:
def format_result(results: list[Document]) -> None:
  for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

# 1. Wikipedia Retrievers

In [9]:
wiki_retrievers = WikipediaRetriever(top_k_result=2, lang='en')

In [10]:
# The word livestock was first used between 1650 and 1660, as a compound word combining the words "live" and "stock".[4] In some periods, "cattle" and "livestock" have been used interchangeably
query1 = 'When was the word “livestock” first used, and how is it related to the word “cattle”?'
wiki_result = wiki_retrievers.invoke(query1)

In [27]:
format_result(wiki_result)


--- Result 1 ---
Cattle (Bos taurus) are large, domesticated, bovid ungulates widely kept as livestock. They are prominent modern members of the subfamily Bovinae and the most widespread species of the genus Bos. Mature female cattle are called cows and mature male cattle are bulls. Young female cattle are called heifers, young male cattle are bullocks, and castrated male cattle are known as oxen or steers. 
Cattle are commonly raised for meat, for dairy products, and for leather. As draft animals, they pull carts and farm implements. Cattle are considered sacred animals within Hinduism, and it is illegal to kill them in some Indian states. Small breeds such as the miniature Zebu are kept as pets.
Taurine cattle are widely distributed across Europe and temperate areas of Asia, the Americas, and Australia. Zebus are found mainly in India and tropical areas of Asia, America, and Australia. Sanga cattle are found primarily in sub-Saharan Africa. These types, sometimes classified as separ

# 2. Vector Store Retriever

In [12]:
# Step 1: Your source documents
documents = [
    Document(page_content="Agriculture is the largest employment sector in Bangladesh, making up 14.2 percent of Bangladesh's GDP in 2017 and employing about 42.7 percent of the workforce.[1] As of the financial year 2022 to 2023, the agricultural sector contributed to more than 12% of GDP.[2] The performance of this sector has an overwhelming impact on major macroeconomic objectives like employment generation, poverty alleviation, human resources development, food security, and other economic and social forces."),
    Document(page_content="A plurality of Bangladeshis earn their living from agriculture. Due to a number of factors, Bangladesh's labour-intensive agriculture has achieved steady increases in food grain production despite the often unfavorable weather conditions."),
    Document(page_content="Wheat is the second major food grain of Bangladesh.[15] While historically not a major crop in Bangladesh, domestic wheat production hit a record high of 1.5 million tonnes in 1985, although still accounting for only 7 to 9 percent of total food grain production.[16] Since then, wheat production in Bangladesh has remained stagnant, with annual production of about 1 million tonnes, falling significantly short of the demand of 7 million tonnes.[17] The shortfall is met through imports, which have exceeded 6 million tonnes, amounting to $1.4–$2 billion in imports annually.[18] Wheat imports constitute the majority of imported food grains in the country. About half of Bangladesh's wheat is grown on irrigated land.[16]"),
    Document(page_content="Rice is Bangladesh's primary crop and staple food, dominating agricultural production, employment, nutritional intake, and contributing substantially to national income.[11] Since 2019, Bangladesh ranked as the third-largest producer of rice globally,[12] reaching about 39.1 million tonnes in 2023.[13] Rice is cultivated in three distinct seasons,[11] with the Bangladesh Rice Research Institute playing a significant role in researching and developing methods to improve its production.[14]"),
]

In [17]:
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    persist_directory = 'my_chroma_db',
    collection_name = 'ipl_data1'
)

In [19]:
query2 = "What is Bangladesh's primary crop, how significant is it to the country, and what organization contributes to improving its production?"

vec_retrivers = vectorstore.as_retriever(kwargs = {'k': 2})
vec_result = vec_retrivers.invoke(query2)

In [20]:
vec_result

[Document(id='81f537b6-25e0-4637-aa51-d09dd5f0887c', metadata={}, page_content="Rice is Bangladesh's primary crop and staple food, dominating agricultural production, employment, nutritional intake, and contributing substantially to national income.[11] Since 2019, Bangladesh ranked as the third-largest producer of rice globally,[12] reaching about 39.1 million tonnes in 2023.[13] Rice is cultivated in three distinct seasons,[11] with the Bangladesh Rice Research Institute playing a significant role in researching and developing methods to improve its production.[14]"),
 Document(id='82f4c087-1ca2-4b1c-8d1a-b514518d5ac0', metadata={}, page_content="Wheat is the second major food grain of Bangladesh.[15] While historically not a major crop in Bangladesh, domestic wheat production hit a record high of 1.5 million tonnes in 1985, although still accounting for only 7 to 9 percent of total food grain production.[16] Since then, wheat production in Bangladesh has remained stagnant, with annu

In [21]:
vectorstore.similarity_search(query=query2, k=2)

[Document(id='81f537b6-25e0-4637-aa51-d09dd5f0887c', metadata={}, page_content="Rice is Bangladesh's primary crop and staple food, dominating agricultural production, employment, nutritional intake, and contributing substantially to national income.[11] Since 2019, Bangladesh ranked as the third-largest producer of rice globally,[12] reaching about 39.1 million tonnes in 2023.[13] Rice is cultivated in three distinct seasons,[11] with the Bangladesh Rice Research Institute playing a significant role in researching and developing methods to improve its production.[14]"),
 Document(id='82f4c087-1ca2-4b1c-8d1a-b514518d5ac0', metadata={}, page_content="Wheat is the second major food grain of Bangladesh.[15] While historically not a major crop in Bangladesh, domestic wheat production hit a record high of 1.5 million tonnes in 1985, although still accounting for only 7 to 9 percent of total food grain production.[16] Since then, wheat production in Bangladesh has remained stagnant, with annu

In [28]:
format_result(vec_result)


--- Result 1 ---
Rice is Bangladesh's primary crop and staple food, dominating agricultural production, employment, nutritional intake, and contributing substantially to national income.[11] Since 2019, Bangladesh ranked as the third-largest producer of rice globally,[12] reaching about 39.1 million tonnes in 2023.[13] Rice is cultivated in three distinct seasons,[11] with the Bangladesh Rice Research Institute playing a significant role in researching and developing methods to improve its production.[14]

--- Result 2 ---
Wheat is the second major food grain of Bangladesh.[15] While historically not a major crop in Bangladesh, domestic wheat production hit a record high of 1.5 million tonnes in 1985, although still accounting for only 7 to 9 percent of total food grain production.[16] Since then, wheat production in Bangladesh has remained stagnant, with annual production of about 1 million tonnes, falling significantly short of the demand of 7 million tonnes.[17] The shortfall is me

# 3. MMR

In [29]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [32]:
mmrstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [40]:
mmr_retrievers = mmrstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {'k': 3, "fetch_k": 50, "lambda_mult": 0.25}  # k = top results, lambda_mult = relevance-diversity balance
)

In [41]:
query3= "What is langchain?"
mmr_results = mmr_retrievers.invoke(query3)

In [42]:
format_result(mmr_results)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
MMR helps you get diverse results when doing similarity search.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


# 4. Multiquery Retriever

In [43]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [44]:
mtqstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embedding_model
)

In [45]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm = llm_model
)

NameError: name 'MultiQueryRetriever' is not defined

- **Note:** It is not available right now

- To Learn More About *Retrievers*, Please Visit https://docs.langchain.com/oss/python/integrations/retrievers